In [8]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import OpenAIEmbeddings

C:\Users\Lavanya Rajesh\AppData\Local\Temp\ipykernel_8452\283773098.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


STEP-I : Load the pdf into text format

In [10]:
text_data = PyPDFLoader("Sample.pdf").load()

# Changing Metadata of the document
for page in text_data:
    page.metadata["source"] = "Sample.pdf"

text_data

[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.5 (Windows)', 'creationdate': '2026-08-13T09:24:03+08:00', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'moddate': '2026-08-13T09:25:30+08:00', 'title': 'W2K-All Across Europe-Reformat-Rev2.indd', 'trapped': '/False', 'source': 'Sample.pdf', 'total_pages': 111, 'page': 0, 'page_label': '1'}, page_content='All Across\nEUROPE\nEllen Weisberg and Ken Yoffe'),
 Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.5 (Windows)', 'creationdate': '2026-08-13T09:24:03+08:00', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'moddate': '2026-08-13T09:25:30+08:00', 'title': 'W2K-All Across Europe-Reformat-Rev2.indd', 'trapped': '/False', 'source': 'Sample.pdf', 'total_pages': 111, 'page': 1, 'page_label': '2'}, page_content='Published by Ken Yoffe and Ellen Weisberg \nwww.facepaint.team\nAll Across Europe\nISBN: 978-1-642

STEP-2 : Creating Chunks of the text data

In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

chunks = splitter.split_documents(text_data)
len(chunks)

569

STEP-3 & 4 : Creating Embeddings and Storing them in Vector DB

In [15]:
from langchain_community.vectorstores import Chroma
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
chroma_db = Chroma.from_documents(chunks, embed_model, persist_directory="./chroma_db")

Step-5 : Connection & Retrieval

In [16]:
chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)

C:\Users\Lavanya Rajesh\AppData\Local\Temp\ipykernel_8452\3816121341.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


In [29]:
chroma_db_con.similarity_search("tell me something about France?", k=3)

[Document(metadata={'gts_pdfxversion': 'PDF/X-1:2001', 'page': 88, 'producer': 'Adobe PDF Library 18.0', 'trapped': '/False', 'page_label': '89', 'creator': 'Adobe InDesign 21.5 (Windows)', 'creationdate': '2026-08-13T09:24:03+08:00', 'total_pages': 111, 'source': 'Sample.pdf', 'title': 'W2K-All Across Europe-Reformat-Rev2.indd', 'moddate': '2026-08-13T09:25:30+08:00', 'gts_pdfxconformance': 'PDF/X-1a:2001'}, page_content='89\nThere is a high standard of living in France\nand a heavy emphasis on education . France'),
 Document(metadata={'total_pages': 111, 'source': 'Sample.pdf', 'title': 'W2K-All Across Europe-Reformat-Rev2.indd', 'creator': 'Adobe InDesign 21.5 (Windows)', 'moddate': '2026-08-13T09:25:30+08:00', 'trapped': '/False', 'page': 87, 'producer': 'Adobe PDF Library 18.0', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'creationdate': '2026-08-13T09:24:03+08:00', 'gts_pdfxversion': 'PDF/X-1:2001', 'page_label': '88'}, page_content='88\nThere is a high standard of living in France\n

STEP-6 : LLM Integration and Answer Generation

In [27]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [31]:
user_query = input("Enter your question: ")

rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
rel_chunks_content = str(rel_chunks_content)

llm.invoke(f"{user_query}, Use the following context to answer the question: {rel_chunks_content}")

AIMessage(content="France is known for its high standard of living and heavy emphasis on education. Paris, the capital city, is a popular tourist destination with many attractions to offer visitors. The country's commitment to education and quality of life make it a desirable place to live and visit.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 99, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EP5115visKryaGPDc5IuqPVdYOa9e', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0af39-ad32-7a43-9c59-80f322bdda1b